## Importar librerías y definir rutas

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
from sentence_transformers import SentenceTransformer
import torch

# Rutas
INPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_limpio")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")
CHUNKS_CSV = METADATA_FOLDER / "chunks.csv"

os.makedirs(METADATA_FOLDER, exist_ok=True)

# Cargar modelo de embeddings multilingüe
# Usamos un modelo balanceado: rápido y eficaz en español
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f"Cargando modelo {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("Modelo cargado.")

# Verificar dispositivo (CPU/GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Usando dispositivo: {device}")

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Cargando modelo paraphrase-multilingual-MiniLM-L12-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado.
Usando dispositivo: cpu


##  Función para dividir texto en chunks con superposición

In [4]:
def chunk_text(text, chunk_size=400, overlap=50):
    # Acceso al tokenizador en versiones >=2.6.0
    tokenizer = model._first_module().tokenizer  # alternativa
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += (chunk_size - overlap)
    return chunks

## Procesar todos los documentos limpios

In [5]:
txt_files = sorted(INPUT_FOLDER.glob("*.txt"))
print(f"Documentos a procesar: {len(txt_files)}")

all_chunks = []

for txt_path in txt_files:
    doc_name = txt_path.stem  # nombre sin extensión
    print(f"Procesando {doc_name}...")
    
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Dividir en chunks
    chunks_texto = chunk_text(text, chunk_size=400, overlap=50)
    print(f"  -> {len(chunks_texto)} chunks generados")
    
    # Para cada chunk, registrar metadatos básicos
    for i, chunk in enumerate(chunks_texto):
        all_chunks.append({
            "documento": doc_name,
            "chunk_id": f"{doc_name}_{i:04d}",
            "texto": chunk,
            "num_tokens": len(model.tokenizer.encode(chunk))
        })

# Crear DataFrame
df_chunks = pd.DataFrame(all_chunks)
print(f"\nTotal de chunks generados: {len(df_chunks)}")
df_chunks.head()

Token indices sequence length is longer than the specified maximum sequence length for this model (2647 > 128). Running this sequence through the model will result in indexing errors


Documentos a procesar: 48
Procesando DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...
  -> 8 chunks generados
Procesando ESTATUTO 2024. 04-09-2024...
  -> 69 chunks generados
Procesando Guía para la organización y orientación del legajo para la docencia ordinaria...
  -> 28 chunks generados
Procesando MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021...
  -> 10 chunks generados
Procesando Modelo de índice del contenido - Legajo...
  -> 2 chunks generados
Procesando Politica Institucional de Inclusión y diversidad cultural v1...
  -> 3 chunks generados
Procesando Politica Institucional de trabajo digno y protección de la persona v.1...
  -> 3 chunks generados
Procesando REGLAMENTO ADMISION 2025.v7...
  -> 64 chunks generados
Procesando REGLAMENTO BECAS 2021 ACTUALIZADO...
  -> 44 chunks generados
Procesando REGLAMENTO CODIGO ETICA INVESTIGACION 2021...
  -> 15 chunks generados
Procesando REGLAMENTO DE ESTUDIOS POSGRADO 2025...
  -> 193 chunks generados
Procesando 

,documento,chunk_id,texto,num_tokens
0,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,Página 1 UNIVERSIDAD PERUANA UNIÓN DIRECTIVA S...,401
1,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,U. 3.5. Reglamento Interno de Trabajo. 3.6. Re...,402
2,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,un grupo de investigación reconocido por la UP...,402
3,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,o la revisión inicial. 5.1.7 El financiamiento...,402
4,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,APC (siempre respetando el monto máximo). La d...,402


## Generar embeddings para todos los chunks

In [6]:
print("Generando embeddings...")
# La función encode de sentence-transformers acepta una lista de strings y devuelve un array numpy
# Mostramos barra de progreso
from tqdm.auto import tqdm
embeddings = model.encode(df_chunks['texto'].tolist(), 
                          show_progress_bar=True,
                          batch_size=32)  # ajusta batch según tu RAM

# Guardar embeddings como lista de numpy arrays (para luego cargar)
# O guardar en formato numpy
embeddings_list = [emb.tolist() for emb in embeddings]
df_chunks['embedding'] = embeddings_list

print(f"Embeddings generados. Dimensión: {embeddings[0].shape}")

Generando embeddings...


Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Embeddings generados. Dimensión: (384,)


## Guardar DataFrame con chunks y embeddings

In [7]:
# Convertir la columna de embeddings a string para guardar en CSV (no es ideal pero simple)
# Opción mejor: guardar como archivo numpy separado y referenciar desde CSV
# Aquí usaremos un CSV sin los embeddings, y guardaremos los embeddings en .npy

# Guardar DataFrame sin embeddings (solo metadatos)
df_metadata = df_chunks[['documento', 'chunk_id', 'texto', 'num_tokens']]
df_metadata.to_csv(CHUNKS_CSV, index=False, encoding='utf-8')
print(f"Metadatos de chunks guardados en {CHUNKS_CSV}")

# Guardar embeddings como archivo numpy
EMBEDDINGS_NPY = METADATA_FOLDER / "embeddings.npy"
np.save(EMBEDDINGS_NPY, np.array(embeddings, dtype=np.float32))
print(f"Embeddings guardados en {EMBEDDINGS_NPY} (shape: {np.array(embeddings).shape})")

# Guardar también la lista de chunk_ids para correspondencia
CHUNK_IDS_NPY = METADATA_FOLDER / "chunk_ids.npy"
np.save(CHUNK_IDS_NPY, df_chunks['chunk_id'].values)
print(f"Chunk IDs guardados en {CHUNK_IDS_NPY}")

Metadatos de chunks guardados en /home/jupyteruser/work/corpus_upeu/metadatos/chunks.csv
Embeddings guardados en /home/jupyteruser/work/corpus_upeu/metadatos/embeddings.npy (shape: (2410, 384))
Chunk IDs guardados en /home/jupyteruser/work/corpus_upeu/metadatos/chunk_ids.npy


## Resumen del corpus

In [8]:
print("Resumen del corpus chunkificado:")
print(f"  Documentos originales: {len(txt_files)}")
print(f"  Chunks totales: {len(df_chunks)}")
print(f"  Tokens promedio por chunk: {df_chunks['num_tokens'].mean():.1f}")
print(f"  Chunks por documento:")
for doc in sorted(df_chunks['documento'].unique()):
    count = df_chunks[df_chunks['documento'] == doc].shape[0]
    print(f"    - {doc}: {count}")

Resumen del corpus chunkificado:
  Documentos originales: 48
  Chunks totales: 2410
  Tokens promedio por chunk: 397.1
  Chunks por documento:
    - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025: 8
    - ESTATUTO 2024. 04-09-2024: 69
    - Guía para la organización y orientación del legajo para la docencia ordinaria: 28
    - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021: 10
    - Modelo de índice del contenido - Legajo: 2
    - Politica Institucional de Inclusión y diversidad cultural v1: 3
    - Politica Institucional de trabajo digno y protección de la persona v.1: 3
    - REGLAMENTO ADMISION 2025.v7: 64
    - REGLAMENTO BECAS 2021 ACTUALIZADO: 44
    - REGLAMENTO CODIGO ETICA INVESTIGACION 2021: 15
    - REGLAMENTO DE ESTUDIOS POSGRADO 2025: 193
    - REGLAMENTO DE ESTUDIOS V5_2025: 227
    - REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4: 26
    - REGLAMENTO DOCENCIA ORDINARIA v3.5: 45
    - REGLAMENTO ESTUDIANTE UNIONISTA V3: 106
    - REGLAMENTO GENERAL U